<a href="https://colab.research.google.com/github/oleg61/--AI--/blob/Learn_AI/%D0%97%D0%B0%D0%B4%D0%B0%D1%87%D0%B0_3_1_%D1%80%D0%B5%D1%88%D0%B5%D0%BD%D0%B8%D0%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-openai openai -q

In [1]:
pip install -U langchain-gigachat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 2.5 MB/s eta 0:00:00


In [ ]:
from getpass import getpass

## Если используете ключ от OpenAI, запустите эту ячейку 👇

In [ ]:
import os
from langchain_openai import ChatOpenAI


# os.environ['OPENAI_API_KEY'] = "Введите ваш OpenAI API ключ"
os.environ['OPENAI_API_KEY'] = getpass(prompt='Введите ваш OpenAI API ключ')

# Инициализируем языковую модель
llm = ChatOpenAI(temperature=0.0)

Введите ваш OpenAI API ключ··········


## Если используете ключ из курса, запустите эти ячейки 👇

In [ ]:
from langchain_openai import ChatOpenAI
from getpass import getpass

#course_api_key= "Введите ваш API ключ, полученный в боте курса"
course_api_key = getpass(prompt='Введите ваш API ключ, полученный в боте курса')

# инициализируем языковую модель
llm = ChatOpenAI(api_key=course_api_key, model='gpt-4o-mini',
                 base_url="https://aleron-llm.neuraldeep.tech/")

Введите ваш API ключ, полученный в боте курса··········


In [5]:
from langchain_gigachat.chat_models import GigaChat
from getpass import getpass

#course_api_key = getpass(prompt='Введите ваш API ключ от курса: ')

# Инициализируем GigaChat через прокси
llm = GigaChat(
    credentials="MDE5YTUwZGEtOTkyNS03NWVjLWEzOWUtM2M4ZDc5YjI5OTMwOjM2OGJkYjk4LWZlNDctNGUzNS04MzM2LTRlZjIzZGRkN2MxNg==",
    verify_ssl_certs=False,
)


print(llm.invoke("Hello, world!"))

content='Привет, мир!\n\nЭто простой классический пример первой программы на любом языке программирования — демонстрация успешной компиляции и запуска. Ты молодец, что начинаешь знакомство с миром кодинга именно здесь 😊' additional_kwargs={} response_metadata={'token_usage': {'prompt_tokens': 14, 'completion_tokens': 43, 'total_tokens': 57, 'precached_prompt_tokens': 2}, 'model_name': 'GigaChat:2.0.28.2', 'x_headers': {'x-request-id': 'f36f384e-cb62-4676-8def-83f4e32634f1', 'x-session-id': 'a9a7d51e-f96c-4099-85a9-c8ea29c8bb59', 'x-client-id': None}, 'finish_reason': 'stop'} id='f36f384e-cb62-4676-8def-83f4e32634f1' usage_metadata={'output_tokens': 43, 'input_tokens': 14, 'total_tokens': 57, 'input_token_details': {'cache_read': 2}}


💾 Добавим памяти LLM'ке! 🦙







In [6]:
# подгрузим датасет
import pandas as pd

df = pd.read_csv("https://stepik.org/media/attachments/lesson/1084404/dial_df.csv")
df.head()

,dialogue_id,question
0,1,Под каким номером играл Криштиану Роналду в Юв...
1,1,Под каким номером он играл в Манчестер Юнайтед?
2,1,В каком году он был рожден?
3,2,Сколько лет прожил величайший русский поэт Але...
4,2,В каком году он родился?


In [7]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage
import re

# Создадим словарь, который будет мапить session_id с историей диалога этой сессии
store = {}

# Напишем функцию, которая возвращает историю диалога по session ID.
def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

### ваше решение
prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты — ассистент, который отвечает на вопросы строго одним целым числом. Никаких слов, только цифра."),
    MessagesPlaceholder(variable_name="history"),  # Сюда подставится история предыдущих сообщений
    ("human", "{input}")  # Текущий вопрос от пользователя
])

In [8]:
chain = prompt | llm

In [9]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

In [10]:
# ответьте на вопросы с помощью ЛЛМки, не забывайте очищать историю диалога при смене темы
ans = []
for id, q in zip(df['dialogue_id'], df['question']):
    session_id = str(id)
    response = chain_with_history.invoke(
        {"input": q},
        config={"configurable": {"session_id": session_id}}
        )
    raw_answer = response.content.strip()
    match = re.search(r'-?\d+', raw_answer)
    final_answer = int(match.group()) if match else 0
    ans.append(final_answer)


    #ans.append()  здесь инференс ЛЛМки
    #break # уберем break, когда убедимся, что работает на одном примере

In [ ]:
final_answer = len(q)  # или len(q) % 100, или любое число
ans.append(final_answer)

In [11]:
df['answer'] = ans # запишем предсказания в датафрейм
df.to_csv('solution.csv', index=False) # сохраним решение в csv

In [12]:
print(len(df))

18


In [13]:
print(len(ans))

18
